# Customer Churn Prediction

**Objective:** Build models to predict customer churn using the provided Telco Customer Churn dataset and complete every task described in the project PDF.

## 1) Data loading & initial inspection

We load the CSV provided (`WA_Fn-UseC_-Telco-Customer-Churn.csv`) and display top rows, info, and missing values.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, auc, classification_report

# Load data
csv_path = '/mnt/data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(csv_path)

print('Shape:', df.shape)
display(df.head())
display(df.info())
display(df.describe(include='all').T)
display(df.isnull().sum())


## 2) Data cleaning & preprocessing
Handle missing values, convert types (e.g., `TotalCharges` may have spaces), and prepare the dataset for modeling.

In [ ]:
data = df.copy()

data['TotalCharges'] = pd.to_numeric(data['TotalCharges'], errors='coerce')
print('Missing TotalCharges count:', data['TotalCharges'].isnull().sum())

# Drop rows with missing TotalCharges (commonly zero-tenure rows)
data = data.dropna(subset=['TotalCharges']).reset_index(drop=True)
print('Shape after dropping missing TotalCharges:', data.shape)

# Drop customerID
if 'customerID' in data.columns:
    data = data.drop(columns=['customerID'])

# Convert Churn
data['Churn'] = data['Churn'].map({'Yes':1, 'No':0})

numeric_cols = ['tenure','MonthlyCharges','TotalCharges']
for c in numeric_cols:
    data[c] = pd.to_numeric(data[c])

cat_cols = data.select_dtypes(include=['object']).columns.tolist()
print('Categorical columns:', cat_cols)
display(data.head())
display(data.info())


## 3) Exploratory Data Analysis (EDA)
Visualize relationships between churn and key categorical variables and show a correlation heatmap for numeric features.

In [ ]:
for col in ['Contract','PaymentMethod','InternetService']:
    plt.figure(figsize=(6,4))
    cat_rates = data.groupby(col)['Churn'].mean().sort_values(ascending=False)
    plt.bar(cat_rates.index, cat_rates.values)
    plt.title(f'Churn rate by {col}')
    plt.ylabel('Churn rate')
    plt.xlabel(col)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

num = ['tenure','MonthlyCharges','TotalCharges','Churn']
corr = data[num].corr()
plt.figure(figsize=(5,4))
plt.imshow(corr, interpolation='nearest', cmap='coolwarm')
plt.colorbar()
plt.xticks(range(len(num)), num, rotation=45)
plt.yticks(range(len(num)), num)
for i in range(len(num)):
    for j in range(len(num)):
        plt.text(j, i, f'{corr.iloc[i,j]:.2f}', ha='center', va='center', color='k')
plt.title('Correlation matrix (numeric features)')
plt.tight_layout()
plt.show()


## 4) Encoding & Feature Engineering
Encode categorical variables (binary and one-hot) and prepare X, y for modeling.

In [ ]:
# Identify categorical columns
cat_cols = data.select_dtypes(include=['object']).columns.tolist()

# Binary candidates
binary_candidates = [c for c in cat_cols if data[c].nunique() == 2]
print('Binary candidate columns:', binary_candidates)

data_encoded = data.copy()
for c in binary_candidates:
    if set(data_encoded[c].unique()) <= set(['Yes','No']):
        data_encoded[c] = data_encoded[c].map({'Yes':1,'No':0})
    elif set(data_encoded[c].unique()) <= set(['Male','Female']):
        data_encoded[c] = data_encoded[c].map({'Male':1,'Female':0})

remaining_cat = [c for c in cat_cols if data_encoded[c].dtype=='object']
print('Remaining categorical columns to one-hot encode:', remaining_cat)
data_encoded = pd.get_dummies(data_encoded, columns=remaining_cat, drop_first=True)
print('Shape after encoding:', data_encoded.shape)
display(data_encoded.head())


## 5) Feature importance
Train a RandomForest to rank features by importance.

In [ ]:
X = data_encoded.drop(columns=['Churn'])
y = data_encoded['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
display(importances.head(30))

top_n = 15
plt.figure(figsize=(8,6))
plt.barh(importances.index[:top_n][::-1], importances.values[:top_n][::-1])
plt.title('Top feature importances (RandomForest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## 6) Model training & evaluation
Train Logistic Regression and RandomForest models, evaluate using multiple metrics, and plot ROC curves.

In [ ]:
# Scale numeric features
numeric_features = ['tenure','MonthlyCharges','TotalCharges']
scaler = StandardScaler()
X_train_sc = X_train.copy()
X_test_sc = X_test.copy()
X_train_sc[numeric_features] = scaler.fit_transform(X_train_sc[numeric_features])
X_test_sc[numeric_features] = scaler.transform(X_test_sc[numeric_features])

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)
y_proba_lr = lr.predict_proba(X_test_sc)[:,1]

# Random Forest predictions
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:,1]

# Evaluation
def evaluate_model(y_true, y_pred, y_proba, model_name='Model'):
    print('---', model_name, '---')
    print('Accuracy:', accuracy_score(y_true, y_pred))
    print('Precision:', precision_score(y_true, y_pred))
    print('Recall:', recall_score(y_true, y_pred))
    print('F1-score:', f1_score(y_true, y_pred))
    try:
        print('ROC AUC:', roc_auc_score(y_true, y_proba))
    except:
        pass
    print('Confusion Matrix:')
    print(confusion_matrix(y_true, y_pred))
    print('\nClassification Report:\n', classification_report(y_true, y_pred))

evaluate_model(y_test, y_pred_lr, y_proba_lr, 'Logistic Regression')
evaluate_model(y_test, y_pred_rf, y_proba_rf, 'Random Forest')

# ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_proba_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
auc_lr = auc(fpr_lr, tpr_lr)
auc_rf = auc(fpr_rf, tpr_rf)

plt.figure(figsize=(6,5))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic (AUC={auc_lr:.2f})')
plt.plot(fpr_rf, tpr_rf, label=f'RandomForest (AUC={auc_rf:.2f})')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()


## 7) Conclusions & retention strategies
Summarize findings and propose retention approaches based on model insights.

**Key takeaways (example):**

- Contract type (Month-to-month) and Payment method may strongly influence churn. Customers on month-to-month contracts have higher churn rates — offering incentives for longer-term contracts can help.
- Higher MonthlyCharges and low tenure are associated with higher churn; consider targeted discounts or customer success outreach for high-risk segments.
- Feature importance from RandomForest shows the most influential variables which should be prioritized in retention campaigns.

**Retention strategies:**

1. Offer discounts or incentives for switching from month-to-month to 1- or 2-year contracts.
2. Improve onboarding and proactive outreach for new customers (low tenure) — e.g., welcome calls, how-to guides, early check-ins.
3. Provide flexible payment options and remind customers before payment issues occur.
4. Create churn-risk score in CRM (use model probabilities) and trigger retention offers automatically.
